# 🌦️ Notebook 01: Data Cleaning & Preprocessing

## Overview
This notebook loads the raw Global Weather Repository dataset and applies a comprehensive 6-step cleaning pipeline:
1. Parse dates & sort by time
2. Extract temporal features (year, month, season, day_of_week)
3. Handle missing values (per-city median imputation)
4. Remove/clip outliers (IQR method)
5. Engineer derived features (heat index, wind chill, etc.)
6. Normalize numeric features

**Output:** `data/weather_cleaned.csv` (ready for EDA)

---

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.10.9-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached matplotlib-3.10.9-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl (232 kB)

   ---------------------------------------- 0/5 [contourpy]
   ---------------------------------------- 0/5 [contourpy]
   -------- ------------------------------- 1/5 [scikit-learn]
   -------- ------------------------------- 1/5 [scikit-learn]
   -------- ------------------------------- 1/5 [scikit-learn]
   -------- --

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Project\\weather prediction\\.venv\\Lib\\site-packages\\pandas\\tests\\indexes\\numeric\\test_astype.py'
Check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Import Libraries & Configure Logging

In [2]:
import pandas as pd
import numpy as np
import logging
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## Step 2: Load Raw Dataset

In [4]:
!pip install kagglehub

  Using cached kagglehub-1.0.1-py3-none-any.whl.metadata (40 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.15-py3-none-any.whl.metadata (7.7 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
Using cached kagglehub-1.0.1-py3-none-any.whl (70 kB)
Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl (437 kB)
Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl (156 kB)
Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl (159 kB)
Using cached idna-3.15-py3-none-any.whl (72 kB)
Using cached certifi-2026.4.22-py3-none-any.whl (135 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)

   ----------------------------------------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nelgiriyewithana/global-weather-repository")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\shami\.cache\kagglehub\datasets\nelgiriyewithana\global-weather-repository\versions\955


In [6]:
# Load the raw CSV
df = pd.read_csv("C:\\Users\\shami\\.cache\\kagglehub\\datasets\\nelgiriyewithana\\global-weather-repository\\versions\\955\\GlobalWeatherRepository.csv")

# Initial inspection
print(f"🔍 Dataset Shape: {df.shape}")
print(f"\n📋 Column Names & Types:")
print(df.dtypes)
print(f"\n📊 First 5 rows:")
df.head()

🔍 Dataset Shape: (141508, 41)

📋 Column Names & Types:
country                             str
location_name                       str
latitude                        float64
longitude                       float64
timezone                            str
last_updated_epoch                int64
last_updated                        str
temperature_celsius             float64
temperature_fahrenheit          float64
condition_text                      str
wind_mph                        float64
wind_kph                        float64
wind_degree                       int64
wind_direction                      str
pressure_mb                     float64
pressure_in                     float64
precip_mm                       float64
precip_in                       float64
humidity                          int64
cloud                             int64
feels_like_celsius              float64
feels_like_fahrenheit           float64
visibility_km                   float64
visibility_miles         

,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [7]:
# Data summary statistics
print(f"📈 Descriptive Statistics:")
print(df.describe())

print(f"\n🌍 Geographic Coverage:")
print(f"   Unique countries: {df['country'].nunique()}")
print(f"   Unique cities: {df['location_name'].nunique()}")
print(f"   Top 10 countries: {df['country'].value_counts().head(10).to_dict()}")

📈 Descriptive Statistics:
            latitude      longitude  last_updated_epoch  temperature_celsius  \
count  141508.000000  141508.000000        1.415080e+05        141508.000000   
mean       19.217035      21.938197        1.747342e+09            21.241554   
std        24.412767      65.783410        1.817776e+07             9.646620   
min       -41.300000    -175.200000        1.715849e+09           -29.800000   
25%         4.050300      -6.836100        1.731661e+09            15.600000   
50%        17.250000      23.236100        1.747386e+09            23.700000   
75%        40.400000      49.882200        1.763019e+09            28.000000   
max        64.150000     179.220000        1.778828e+09            79.300000   

       temperature_fahrenheit       wind_mph       wind_kph    wind_degree  \
count           141508.000000  141508.000000  141508.000000  141508.000000   
mean                70.236594       7.985414      12.854982     168.874290   
std                

In [8]:
# Missing values analysis
print(f"❌ Missing Values:")
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(missing)
print(f"\n% Missing:")
print((missing / len(df) * 100).round(2))

❌ Missing Values:
Series([], dtype: int64)

% Missing:
Series([], dtype: float64)


## Step 3: Cleaning Pipeline

In [9]:
def clean_weather_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Comprehensive data cleaning and feature engineering pipeline.
    
    Args:
        df: Raw weather DataFrame
        
    Returns:
        Cleaned and feature-engineered DataFrame
    """
    df = df.copy()
    
    # ── 1. Parse Dates ────────────────────────────────────────────────────────
    logger.info("Step 1: Parsing dates...")
    df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")
    initial_rows = df.shape[0]
    df = df.dropna(subset=["last_updated"])
    dropped = initial_rows - df.shape[0]
    if dropped > 0:
        logger.warning(f"   Dropped {dropped} rows with unparseable dates")
    
    df = df.sort_values("last_updated").reset_index(drop=True)
    logger.info(f"✅ Date range: {df['last_updated'].min()} → {df['last_updated'].max()}")
    
    # ── 2. Extract Time Features ──────────────────────────────────────────────
    logger.info("Step 2: Extracting temporal features...")
    df["year"]        = df["last_updated"].dt.year
    df["month"]       = df["last_updated"].dt.month
    df["day"]         = df["last_updated"].dt.day
    df["hour"]        = df["last_updated"].dt.hour
    df["day_of_week"] = df["last_updated"].dt.dayofweek
    df["is_weekend"]  = df["day_of_week"].isin([5, 6]).astype(int)
    df["season"]      = df["month"].map({
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring",  4: "Spring", 5: "Spring",
        6: "Summer",  7: "Summer", 8: "Summer",
        9: "Autumn", 10: "Autumn", 11: "Autumn"
    })
    logger.info("✅ Temporal features extracted")
    
    # ── 3. Handle Missing Values ──────────────────────────────────────────────
    logger.info("Step 3: Handling missing values...")
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
    
    # Fill numeric with median per city
    for col in numeric_cols:
        df[col] = df.groupby("location_name")[col].transform(
            lambda x: x.fillna(x.median())
        )
    
    # Global median for remaining NaNs
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    
    # Fill categorical with mode
    for col in categorical_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mode()[0])
    
    logger.info(f"✅ Missing values handled (remaining: {df.isnull().sum().sum()})")
    
    # ── 4. Outlier Handling (IQR method) ──────────────────────────────────────
    logger.info("Step 4: Clipping outliers...")
    key_cols = ["temperature_celsius", "precip_mm", "wind_kph", "humidity"]
    key_cols = [c for c in key_cols if c in df.columns]
    
    for col in key_cols:
        Q1 = df[col].quantile(0.01)
        Q3 = df[col].quantile(0.99)
        IQR = Q3 - Q1
        lower = Q1 - 3 * IQR
        upper = Q3 + 3 * IQR
        outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
        if outlier_count > 0:
            logger.info(f"   {col}: {outlier_count} outliers clipped")
        df[col] = df[col].clip(lower, upper)
    
    # ── 5. Feature Engineering ─────────────────────────────────────────────────
    logger.info("Step 5: Engineering derived features...")
    if "temperature_celsius" in df.columns and "humidity" in df.columns:
        df["temp_humidity_index"] = df["temperature_celsius"] * df["humidity"] / 100
        df["heat_index"] = df["temperature_celsius"] + 0.33 * (df["humidity"] / 100 * 6.105) - 4
    
    if "temperature_celsius" in df.columns and "wind_kph" in df.columns:
        df["wind_chill"] = 13.12 + 0.6215 * df["temperature_celsius"] - 11.37 * (df["wind_kph"] ** 0.16)
    
    if "temperature_celsius" in df.columns and "feelslike_celsius" in df.columns:
        df["temp_feels_like_diff"] = df["temperature_celsius"] - df["feelslike_celsius"]
    
    if "precip_mm" in df.columns:
        df["log_precip"] = np.log1p(df["precip_mm"])
    
    logger.info("✅ Derived features created")
    
    # ── 6. Normalize Numeric Features ─────────────────────────────────────────
    logger.info("Step 6: Normalizing numeric features...")
    scaler = MinMaxScaler()
    scale_cols = ["temperature_celsius", "humidity", "wind_kph", "precip_mm",
                  "pressure_mb", "visibility_km", "uv_index"]
    scale_cols = [c for c in scale_cols if c in df.columns]
    
    if scale_cols:
        df[[f"{c}_scaled" for c in scale_cols]] = scaler.fit_transform(df[scale_cols])
        logger.info(f"✅ Normalized {len(scale_cols)} features")
    
    logger.info(f"\n✅ Cleaning complete!")
    logger.info(f"   Final shape: {df.shape}")
    logger.info(f"   Remaining nulls: {df.isnull().sum().sum()}")
    
    return df

print("✅ Cleaning function defined")

✅ Cleaning function defined


## Step 4: Execute Cleaning Pipeline

In [10]:
# Run the cleaning pipeline
df_clean = clean_weather_data(df)

print("\n" + "="*60)
print("CLEANING PIPELINE COMPLETED")
print("="*60)

2026-05-15 20:54:08,323 - INFO - Step 1: Parsing dates...
2026-05-15 20:54:08,407 - INFO - ✅ Date range: 2024-05-16 01:45:00 → 2026-05-15 19:45:00
2026-05-15 20:54:08,408 - INFO - Step 2: Extracting temporal features...
2026-05-15 20:54:08,443 - INFO - ✅ Temporal features extracted
2026-05-15 20:54:08,444 - INFO - Step 3: Handling missing values...
2026-05-15 20:54:11,953 - INFO - ✅ Missing values handled (remaining: 0)
2026-05-15 20:54:11,954 - INFO - Step 4: Clipping outliers...
2026-05-15 20:54:11,974 - INFO -    precip_mm: 78 outliers clipped
2026-05-15 20:54:11,984 - INFO -    wind_kph: 5 outliers clipped
2026-05-15 20:54:12,001 - INFO - Step 5: Engineering derived features...
2026-05-15 20:54:12,012 - INFO - ✅ Derived features created
2026-05-15 20:54:12,015 - INFO - Step 6: Normalizing numeric features...
2026-05-15 20:54:12,040 - INFO - ✅ Normalized 7 features
2026-05-15 20:54:12,041 - INFO - 
✅ Cleaning complete!
2026-05-15 20:54:12,041 - INFO -    Final shape: (141508, 59)
20


CLEANING PIPELINE COMPLETED


## Step 5: Verify Cleaned Data

In [11]:
# Display cleaned data summary
print(f"✅ Cleaned Dataset Summary:")
print(f"   Shape: {df_clean.shape}")
print(f"   Missing values: {df_clean.isnull().sum().sum()}")
print(f"   Date range: {df_clean['last_updated'].min()} to {df_clean['last_updated'].max()}")
print(f"\n📊 Column Summary:")
print(df_clean.info())

✅ Cleaned Dataset Summary:
   Shape: (141508, 59)
   Missing values: 0
   Date range: 2024-05-16 01:45:00 to 2026-05-15 19:45:00

📊 Column Summary:
<class 'pandas.DataFrame'>
RangeIndex: 141508 entries, 0 to 141507
Data columns (total 59 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   country                       141508 non-null  str           
 1   location_name                 141508 non-null  str           
 2   latitude                      141508 non-null  float64       
 3   longitude                     141508 non-null  float64       
 4   timezone                      141508 non-null  str           
 5   last_updated_epoch            141508 non-null  int64         
 6   last_updated                  141508 non-null  datetime64[us]
 7   temperature_celsius           141508 non-null  float64       
 8   temperature_fahrenheit        141508 non-null  float64       
 9   condition_

In [12]:
# Sample of cleaned data
print(f"\n🔍 Sample of Cleaned Data (first 5 rows):")
df_clean.head()


🔍 Sample of Cleaned Data (first 5 rows):


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,heat_index,wind_chill,log_precip,temperature_celsius_scaled,humidity_scaled,wind_kph_scaled,precip_mm_scaled,pressure_mb_scaled,visibility_km_scaled,uv_index_scaled
0,United States of America,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,2024-05-16 01:45:00,16.1,61.0,Clear,...,13.268497,7.674962,0.000000,0.420715,0.571429,0.023669,0.000000,0.031569,0.50000,0.06135
1,Mexico,Mexico City,19.43,-99.13,America/Mexico_City,1715849100,2024-05-16 02:45:00,20.8,69.4,Clear,...,17.746886,9.408926,0.000000,0.463795,0.459184,0.053254,0.000000,0.032054,0.31250,0.06135
2,Costa Rica,San Juan,9.97,-84.08,America/Costa_Rica,1715849100,2024-05-16 02:45:00,21.0,69.8,Fog,...,19.014650,12.215246,0.157004,0.465628,1.000000,0.000000,0.020047,0.033511,0.21875,0.06135
3,El Salvador,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,2024-05-16 02:45:00,26.0,78.8,Moderate or heavy rain with thunder,...,23.893771,15.322746,0.262364,0.511457,0.938776,0.000000,0.035377,0.030597,0.31250,0.06135
4,Guatemala,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,2024-05-16 02:45:00,20.0,68.0,Mist,...,17.772892,6.905614,0.086178,0.456462,0.877551,0.136095,0.010613,0.034968,0.15625,0.06135


In [13]:
# Descriptive statistics
print(f"\n📈 Descriptive Statistics (Cleaned):")
df_clean.describe().round(2)


📈 Descriptive Statistics (Cleaned):


,latitude,longitude,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,wind_mph,wind_kph,wind_degree,pressure_mb,...,heat_index,wind_chill,log_precip,temperature_celsius_scaled,humidity_scaled,wind_kph_scaled,precip_mm_scaled,pressure_mb_scaled,visibility_km_scaled,uv_index_scaled
count,141508.00,141508.00,1.415080e+05,141508,141508.00,141508.00,141508.00,141508.00,141508.00,141508.00,...,141508.00,141508.00,141508.00,141508.00,141508.00,141508.00,141508.00,141508.00,141508.00,141508.00
mean,19.22,21.94,1.747342e+09,2025-05-15 22:53:05.552760,21.24,70.24,7.99,12.83,168.87,1014.04,...,18.59,9.69,0.08,0.47,0.66,0.07,0.02,0.03,0.30,0.20
min,-41.30,-175.20,1.715849e+09,2024-05-16 01:45:00,-29.80,-21.60,2.20,3.60,1.00,947.00,...,-32.67,-20.74,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,4.05,-6.84,1.731661e+09,2024-11-15 10:00:00,15.60,60.00,3.80,6.10,80.00,1010.00,...,13.03,6.33,0.00,0.42,0.50,0.02,0.00,0.03,0.31,0.01
50%,17.25,23.24,1.747386e+09,2025-05-16 03:00:00,23.70,74.70,6.70,10.80,161.00,1013.00,...,21.09,10.96,0.00,0.49,0.71,0.05,0.00,0.03,0.31,0.11
75%,40.40,49.88,1.763019e+09,2025-11-13 09:30:00,28.00,82.30,11.00,17.60,256.00,1018.00,...,25.11,13.72,0.02,0.53,0.86,0.10,0.00,0.03,0.31,0.37
max,64.15,179.22,1.778828e+09,2026-05-15 19:45:00,79.30,174.70,1841.20,138.80,360.00,3006.00,...,75.44,44.65,2.25,1.00,1.00,1.00,1.00,1.00,1.00,1.00
std,24.41,65.78,1.817776e+07,NaN,9.65,17.36,7.14,8.34,103.69,10.21,...,9.49,5.99,0.23,0.09,0.24,0.06,0.06,0.00,0.08,0.22


In [ ]:
# Verify no missing values
missing_check = df_clean.isnull().sum()
if missing_check.sum() == 0:
    print("✅ SUCCESS: No missing values in cleaned dataset")
else:
    print(f"⚠️ WARNING: {missing_check.sum()} missing values remain:")
    print(missing_check[missing_check > 0])

## Step 6: Save Cleaned Data

In [ ]:
# Save cleaned dataset
output_path = "../data/weather_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved to: {output_path}")
print(f"   File size: {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   Rows: {len(df_clean):,}")
print(f"   Columns: {len(df_clean.columns)}")
print(f"\n🎉 Notebook 01 Complete! Ready for EDA.")